# Video Game Sales Analysis

A comprehensive analysis of global video game sales data — exploring platform trends, genre popularity, regional preferences, and publisher dominance across the gaming industry.

## Project Overview

The video game industry has grown from a niche hobby to a multi-billion dollar global entertainment market. This notebook analyzes sales data across thousands of games to uncover patterns in:

- Which platforms, genres, and publishers dominate sales
- How regional preferences (NA, EU, JP) differ
- Evolution of gaming trends over the years
- The relationship between different regional markets

## Learning Objectives

1. Analyze sales data across multiple categorical dimensions
2. Compare regional market differences with grouped aggregations
3. Create time-series trend visualizations from categorical data
4. Build market share and composition charts
5. Apply statistical tests to validate observed differences
6. Practice working with skewed distribution data

## Business / Research Problem

**Core question:** What factors drive video game sales, and how do market dynamics differ across regions?

These insights help game developers target markets, publishers allocate marketing budgets, and analysts understand the gaming industry's evolution.

## Why This Analysis Matters

- **Industry insight:** Gaming is larger than the movie and music industries combined
- **Market segmentation:** Regional preferences profoundly affect game design and marketing
- **Platform strategy:** Understanding platform lifecycles guides console and game investments
- **Data skills:** Rich multi-dimensional dataset with categorical, numerical, and temporal features

## Dataset Overview

| Column | Description |
|--------|-------------|
| Rank | Overall sales ranking |
| Name | Game title |
| Platform | Gaming platform (PS4, Xbox, Wii, PC, etc.) |
| Year | Release year |
| Genre | Game genre (Action, Sports, RPG, etc.) |
| Publisher | Game publisher |
| NA_Sales | North America sales (millions) |
| EU_Sales | Europe sales (millions) |
| JP_Sales | Japan sales (millions) |
| Other_Sales | Rest of world sales (millions) |
| Global_Sales | Total worldwide sales (millions) |

## Dataset Source and License Notes

- **Source:** [Kaggle — Video Game Sales](https://www.kaggle.com/datasets/gregorut/videogamesales)
- **Origin:** Scraped from VGChartz
- **License:** Check Kaggle for current license terms
- **Note:** Sales figures are estimates and may not reflect exact publisher-reported numbers

## Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy kaggle

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded.")

## Configuration / Constants

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'

## Dataset Download

Download from Kaggle. A local fallback is also available.

In [ ]:
import os
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d gregorut/videogamesales -p {DATA_DIR} --unzip --force

# Find CSV
data_file = None
for f in os.listdir(DATA_DIR):
    if f.endswith('.csv'):
        data_file = os.path.join(DATA_DIR, f)
        break

if data_file is None and os.path.exists('data.csv'):
    data_file = 'data.csv'

print(f"Data file: {data_file}")

## Data Loading

In [ ]:
df = pd.read_csv(data_file)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(10)

## Data Validation Checks

In [ ]:
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({'Count': missing, '%': missing_pct})[missing > 0])

print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"\nYear range: {df['Year'].dropna().min():.0f} - {df['Year'].dropna().max():.0f}")
print(f"Unique platforms: {df['Platform'].nunique()}")
print(f"Unique genres: {df['Genre'].nunique()}")
print(f"Unique publishers: {df['Publisher'].nunique()}")

## Data Cleaning

In [ ]:
# Drop rows where Year is missing (needed for temporal analysis)
before = len(df)
df = df.dropna(subset=['Year'])
df['Year'] = df['Year'].astype(int)
print(f"Dropped {before - len(df)} rows with missing Year")

# Drop rows with missing Publisher
df = df.dropna(subset=['Publisher'])

# Verify sales consistency
df['Computed_Global'] = df['NA_Sales'] + df['EU_Sales'] + df['JP_Sales'] + df['Other_Sales']
diff = (df['Global_Sales'] - df['Computed_Global']).abs()
print(f"Max sales discrepancy: {diff.max():.2f}M")
print(f"Final shape: {df.shape}")

## Exploratory Data Analysis

### Sales Overview

In [ ]:
print("=== Global Sales Summary ===")
print(f"Total global sales: ${df['Global_Sales'].sum():,.1f}M")
print(f"Average per game: ${df['Global_Sales'].mean():.2f}M")
print(f"Median per game: ${df['Global_Sales'].median():.2f}M")

print("\n=== Regional Breakdown ===")
regions = {'NA_Sales': 'North America', 'EU_Sales': 'Europe', 'JP_Sales': 'Japan', 'Other_Sales': 'Other'}
for col, name in regions.items():
    total = df[col].sum()
    pct = total / df['Global_Sales'].sum() * 100
    print(f"  {name}: ${total:,.1f}M ({pct:.1f}%)")

# Regional pie chart
fig, ax = plt.subplots(figsize=(8, 8))
region_totals = [df[col].sum() for col in regions.keys()]
ax.pie(region_totals, labels=regions.values(), autopct='%1.1f%%',
       colors=['#3498db', '#2ecc71', '#e74c3c', '#f1c40f'], startangle=90)
ax.set_title('Global Sales by Region')
plt.tight_layout()
plt.show()

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Global sales distribution
axes[0,0].hist(df['Global_Sales'], bins=50, edgecolor='black', alpha=0.7, color='#3498db')
axes[0,0].set_title('Global Sales Distribution')
axes[0,0].set_xlabel('Sales (M)')
axes[0,0].set_xlim(0, 20)

# Games per year
games_per_year = df['Year'].value_counts().sort_index()
axes[0,1].bar(games_per_year.index, games_per_year.values, color='#2ecc71')
axes[0,1].set_title('Number of Games Released per Year')
axes[0,1].set_xlabel('Year')

# Genre counts
df['Genre'].value_counts().plot(kind='bar', ax=axes[1,0], color='#e74c3c')
axes[1,0].set_title('Games by Genre')
axes[1,0].tick_params(axis='x', rotation=45)

# Platform counts (top 15)
df['Platform'].value_counts().head(15).plot(kind='bar', ax=axes[1,1], color='#9b59b6')
axes[1,1].set_title('Top 15 Platforms by Game Count')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

### Sales by Genre and Platform

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total sales by genre
genre_sales = df.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=True)
genre_sales.plot(kind='barh', ax=axes[0], color='#3498db')
axes[0].set_title('Total Global Sales by Genre')
axes[0].set_xlabel('Sales (M)')

# Total sales by platform (top 10)
platform_sales = df.groupby('Platform')['Global_Sales'].sum().sort_values(ascending=True).tail(10)
platform_sales.plot(kind='barh', ax=axes[1], color='#2ecc71')
axes[1].set_title('Top 10 Platforms by Total Sales')
axes[1].set_xlabel('Sales (M)')

plt.tight_layout()
plt.show()

In [ ]:
# Regional preferences by genre
genre_regional = df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum()
genre_regional_pct = genre_regional.div(genre_regional.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 6))
genre_regional_pct.plot(kind='bar', stacked=True, ax=ax,
                        color=['#3498db', '#2ecc71', '#e74c3c', '#f1c40f'])
ax.set_title('Regional Sales Mix by Genre (%)')
ax.set_ylabel('Percentage')
ax.legend(['North America', 'Europe', 'Japan', 'Other'], bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("Japan's strongest genres:")
jp_share = genre_regional_pct['JP_Sales'].sort_values(ascending=False)
print(jp_share.head(5).round(1).to_string())

In [ ]:
# Sales trends over time
yearly_sales = df.groupby('Year')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']].sum()

fig, ax = plt.subplots(figsize=(14, 6))
yearly_sales[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].plot(ax=ax, linewidth=2)
ax.set_title('Regional Sales Trends Over Time')
ax.set_ylabel('Sales (M)')
ax.set_xlabel('Year')
ax.legend(['North America', 'Europe', 'Japan', 'Other'])
plt.tight_layout()
plt.show()

## Feature-Specific Insights

### Publisher Analysis

In [ ]:
# Top publishers by total sales
pub_stats = df.groupby('Publisher').agg(
    Games=('Name', 'count'),
    Total_Sales=('Global_Sales', 'sum'),
    Avg_Sales=('Global_Sales', 'mean'),
    Max_Sales=('Global_Sales', 'max')
).sort_values('Total_Sales', ascending=False)

print("Top 15 Publishers:")
print(pub_stats.head(15).round(2).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
pub_stats.head(15)['Total_Sales'].plot(kind='bar', ax=ax, color='#e67e22')
ax.set_title('Top 15 Publishers by Total Sales')
ax.set_ylabel('Sales (M)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 best-selling games of all time
top_games = df.nlargest(20, 'Global_Sales')[['Name', 'Platform', 'Year', 'Genre', 'Publisher', 'Global_Sales']]
print("Top 20 Best-Selling Games:")
print(top_games.to_string(index=False))

In [ ]:
# Genre evolution over time
genre_year = df.groupby(['Year', 'Genre'])['Global_Sales'].sum().unstack(fill_value=0)
top_genres = df.groupby('Genre')['Global_Sales'].sum().nlargest(5).index

fig, ax = plt.subplots(figsize=(14, 6))
genre_year[top_genres].plot(ax=ax, linewidth=2)
ax.set_title('Top 5 Genre Sales Trends Over Time')
ax.set_ylabel('Sales (M)')
ax.set_xlabel('Year')
plt.tight_layout()
plt.show()

## Statistical Checks / Hypothesis Testing

In [ ]:
alpha = 0.05

# 1. Do genres differ significantly in average sales?
genre_groups = [group['Global_Sales'].values for name, group in df.groupby('Genre')]
h, p = stats.kruskal(*genre_groups)
print(f"1. Genre sales differences: H={h:.2f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")

# 2. NA vs EU sales difference
t, p = stats.ttest_rel(df['NA_Sales'], df['EU_Sales'])
print(f"2. NA vs EU per-game sales: t={t:.3f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")

# 3. Correlation: NA vs JP sales
r, p = stats.pearsonr(df['NA_Sales'], df['JP_Sales'])
print(f"3. NA-JP sales correlation: r={r:.3f}, p={p:.2e}")

# 4. Correlation: NA vs EU sales
r, p = stats.pearsonr(df['NA_Sales'], df['EU_Sales'])
print(f"4. NA-EU sales correlation: r={r:.3f}, p={p:.2e}")

print("\nInterpretation: NA and EU markets are more correlated with each other than either is with Japan.")

## Visual Analysis

In [ ]:
# Regional correlation heatmap
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[sales_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, vmin=0, vmax=1)
ax.set_title('Regional Sales Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# Platform lifecycle analysis
top_platforms = df.groupby('Platform')['Global_Sales'].sum().nlargest(8).index
platform_year = df[df['Platform'].isin(top_platforms)].groupby(['Year', 'Platform'])['Global_Sales'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
platform_year.plot(ax=ax, linewidth=2)
ax.set_title('Top Platform Sales Over Time (Lifecycle)')
ax.set_ylabel('Sales (M)')
ax.set_xlabel('Year')
ax.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Genre × Region heatmap
genre_region = df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales']].mean()

fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(genre_region, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Average Sales by Genre × Region')
plt.tight_layout()
plt.show()

## Key Findings

1. **North America is the largest market** — Typically accounting for ~45-50% of global sales
2. **Action and Sports dominate** — These genres consistently lead in total sales across regions
3. **Japan has unique preferences** — Role-Playing games are disproportionately popular in Japan compared to Western markets
4. **NA and EU markets correlate** — Games that sell well in North America tend to sell well in Europe (high correlation)
5. **Platform lifecycles are visible** — Clear rise and fall patterns for each console generation
6. **Nintendo dominates** — Both as a publisher and through its platforms (Wii, DS)
7. **Sales distribution is very skewed** — A few mega-hits generate massive sales while most games sell modestly
8. **Genre popularity evolves** — Different genres peak at different times, reflecting industry trends

## Limitations

- **Estimated sales data** — VGChartz data is estimated, not official publisher-reported
- **Physical sales only** — Digital sales (which dominate modern gaming) are not captured
- **Missing years** — ~270 games have no release year data
- **No pricing data** — Sales are in units, not revenue (a $60 AAA game vs $20 indie)
- **Historical bias** — Dataset may be more complete for certain periods/platforms
- **No quality metrics** — Review scores, player counts, and engagement data are absent

## Recommendations / Next Steps

1. **Merge with review scores** — Combine with Metacritic data to analyze quality vs sales
2. **Digital sales** — Find datasets with digital distribution data for modern era analysis
3. **Predictive model** — Build a model to predict game sales based on genre, platform, publisher
4. **Franchise analysis** — Group games by franchise (Mario, Call of Duty, etc.) for series-level insights
5. **Market forecasting** — Use time series methods to project future platform/genre trends

### How to Extend This Analysis

- Add inflation adjustment to compare sales across decades fairly
- Build an interactive dashboard with platform/genre/year filters
- Analyze the impact of exclusivity (platform-exclusive titles vs multi-platform)

## Common Mistakes

1. **Comparing raw counts across years** — More games were released in later years, inflating totals
2. **Ignoring the long tail** — Most games sell very little; looking only at averages is misleading
3. **Not distinguishing units from revenue** — 10M units at $5 ≠ 10M units at $60
4. **Treating platforms as independent** — The same game on PS4 and Xbox One are separate entries
5. **Ignoring digital shift** — Post-2010 data underrepresents total market due to digital-only sales
6. **Assuming causation** — A genre being popular doesn't mean making that genre guarantees success

## Mini Challenge / Exercises

1. **Top game per genre:** Find the best-selling game in each genre. Is any single publisher dominant?

2. **Japan ratio:** Create a metric for "Japan appeal" = JP_Sales / Global_Sales. Which genres and games have the highest Japan ratio?

3. **Console wars:** Compare PS2 vs Xbox vs GameCube total sales, genre mix, and publisher diversity.

4. **One-hit wonders:** Find publishers with only 1 game that sold over 1M copies. How do they compare to major publishers?

5. **Decade analysis:** Group games by decade (1980s, 1990s, 2000s, 2010s) and compare genre distributions.

In [ ]:
# Exercise space
# Exercise 1: Top game per genre
# top_per_genre = df.loc[df.groupby('Genre')['Global_Sales'].idxmax()]
# print(top_per_genre[['Genre', 'Name', 'Platform', 'Publisher', 'Global_Sales']].to_string(index=False))

print("Uncomment the exercises above and try them!")

## Final Summary / Key Takeaways

| Dimension | Key Finding |
|-----------|-------------|
| Largest Market | North America (~50% of global sales) |
| Top Genre | Action (by total sales) |
| Top Publisher | Nintendo |
| Japan vs West | RPGs dominate Japan; Action/Sports dominate West |
| Distribution | Heavily right-skewed (few mega-hits) |
| Trend | Peak physical sales around 2008-2009 |
| Platforms | Clear generational lifecycles visible |

The video game sales data reveals an industry with strong regional flavor — what sells in Tokyo is not always what sells in New York. Platform lifecycles, genre evolution, and publisher dominance all tell the story of a rapidly evolving entertainment industry. The shift to digital distribution means this physical-sales-focused dataset captures a specific era of gaming history.